In [ ]:
from bs4 import BeautifulSoup
import re
import json
import os
from pathlib import Path

In [ ]:
root_folder = 'cpgs'

html_files = []
file_names = []
for entry in os.listdir(root_folder):
    level1_path = os.path.join(root_folder, entry)
    if os.path.isdir(level1_path):
        for file in os.listdir(level1_path):
            file_path = os.path.join(level1_path, file)
            if os.path.isfile(file_path) and file.endswith('.html'):
                html_files.append(file_path)
                file_names.append(file)

In [ ]:
output_dir = Path("CPG_processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for j in range(len(html_files)):
    with open(html_files[j], "r") as f:
        soup = BeautifulSoup(f, "html.parser")
    script_text = next((script.text for script in soup.find_all("script") if "elements:" in script.text), "")
    match = re.search(r'elements\s*:\s*(\[\s*(?:.|\n)*?\])', script_text)
    graph_data = []

    if match:
        array_str = match.group(1)

        try:
            graph_data = json.loads(array_str)
        except json.JSONDecodeError as e:
            print("JSON parse error:", e)
    else:
        print("No graph array found.")
    graph_dict = {"edges":[]}
    node_2_id = {}

    for i in range(len(graph_data)):
        if 'source' in graph_data[i]['data'].keys():
            rel = ''
            if graph_data[i]['data']['color'] == '#3498db':
                rel = 'excitatory'
            elif graph_data[i]['data']['color'] == '#e74c3c':
                rel = 'inhibitory'
            graph_dict["edges"].append([graph_data[i]['data']['source'], rel,graph_data[i]['data']['target']])

    with open(os.path.join(output_dir, (file_names[j].split('.')[0] + '.json')), "w") as f:
        json.dump(graph_dict, f)